# Phase 4 — Model B training
This notebook resolves `SOURCE_REF`, detaches that exact refactor commit, and never loads ViLexNorm Test.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'

if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPOSITORY_URL, str(REPO)], check=True)
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', source_commit], check=True)
print(f'Running refactored training source at {source_commit}')

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt

import platform, torch, transformers
print(platform.python_version(), torch.__version__, transformers.__version__)
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator'

## Paths
Update these three input paths to match the attached Kaggle datasets.

In [ ]:
DATA = Path('/kaggle/input/visolexnorm-phase3')
MODEL_A = Path('/kaggle/input/visolexnorm-model-a/model_a')
WORK = Path('/kaggle/working')

In [ ]:
!python -m pytest tests/training/test_mixtures.py tests/contract/test_model_b_contracts.py -q
!python -m scripts.training build-mixture --model model_b --repo-root {DATA} --config {REPO}/configs/model_b_config.json --phase3-manifest {DATA}/outputs/phase3_manifest.json --output {WORK}/training_mixture_manifest.json

## Smoke gate: exactly 200 gold + 200 pseudo

In [ ]:
!python -m scripts.training train --model model_b --model-a-checkpoint {MODEL_A} --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --work-dir {WORK}/smoke --smoke-test
import json
smoke = json.load(open(WORK/'smoke/outputs/model_b/smoke_test.json'))
assert smoke['passed'], smoke
assert smoke['composition'] == {'gold': 200, 'pseudo': 200}, smoke
assert smoke['checkpoint_reload'] and smoke['generation_nonempty'], smoke
if not smoke['loss_decreased']:
    print('WARNING: Dev loss did not improve in this tiny smoke run. Pipeline is valid; review before full training.')
smoke

## Full three-epoch run

In [ ]:
!python -m scripts.training train --model model_b --model-a-checkpoint {MODEL_A} --data-dir {DATA}/data/processed --mixture-manifest {WORK}/training_mixture_manifest.json --work-dir {WORK}

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/model_b_artifacts', 'zip', '/kaggle/working', 'outputs/model_b')
shutil.make_archive('/kaggle/working/model_b_checkpoint', 'zip', '/kaggle/working', 'checkpoints/model_b')
print('Download both zip files from Kaggle Output.')